# 05 - Regression Models (Task 6) and Regression Interpretation (Task 8, part 1)

Predict the numerical quality score with Linear Regression, a Decision Tree and a Random Forest.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("../outputs/charts", exist_ok=True)
df = pd.read_csv("../data/wine_quality_cleaned.csv")
print(df.shape)

## 1. Target and predictors
- **Target:** `quality`.
- **Predictors:** the same six variables used in the OLS model (notebook 04), so the statistical model and the machine-learning models can be compared fairly in Task 8.

In [ ]:
features = ["alcohol", "density", "pH", "volatile acidity", "citric acid", "residual sugar"]

X_reg = df[features]
y_reg = df["quality"]

## 2. Train/test split (80/20, fixed random seed)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

## 3. Train the three models

In [ ]:
lr_reg = LinearRegression().fit(X_train, y_train)
dt_reg = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

print("All three regression models trained successfully")

## 4. Predictions on the test set and comparison (MAE, MSE, RMSE, R2)

In [ ]:
reg_models = {"Linear Regression": lr_reg, "Decision Tree": dt_reg, "Random Forest": rf_reg}
reg_predictions = {name: model.predict(X_test) for name, model in reg_models.items()}

rows = []
for name, pred in reg_predictions.items():
    mse = mean_squared_error(y_test, pred)
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2 Score": r2_score(y_test, pred),
    })

reg_comparison = pd.DataFrame(rows)
display(reg_comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=reg_comparison, x="Model", y="RMSE", hue="Model", legend=False, ax=axes[0])
axes[0].set_title("RMSE by model (lower is better)")
axes[0].set_xlabel("Regression model")
sns.barplot(data=reg_comparison, x="Model", y="R2 Score", hue="Model", legend=False, ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("R2 on the test set (higher is better)")
axes[1].set_xlabel("Regression model")
plt.tight_layout()
plt.savefig("../outputs/charts/regression_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Actual versus predicted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, (name, pred) in zip(axes, reg_predictions.items()):
    ax.scatter(y_test, pred, alpha=0.4)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
    ax.set_title(name)
    ax.set_xlabel("Actual wine quality")
axes[0].set_ylabel("Predicted wine quality")
plt.suptitle("Actual vs Predicted Wine Quality (test set)", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/charts/regression_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Select the best model

In [ ]:
best_row = reg_comparison.sort_values(by="RMSE", ascending=True).iloc[0]
print("Best model :", best_row["Model"])
print("RMSE       :", round(best_row["RMSE"], 4))
print("MAE        :", round(best_row["MAE"], 4))
print("R2 Score   :", round(best_row["R2 Score"], 4))

**Selection criterion: lowest test RMSE** (it penalises large errors and is in the same unit as quality). The **Random Forest** wins on every metric (MAE 0.583, RMSE 0.762, R2 0.291).

Remarks:
- The margin over Linear Regression is small (RMSE 0.762 vs 0.778). Both predict the quality score to within about 0.6 points on average.
- The **Decision Tree has a negative R2 (-0.36)**: on new data it predicts worse than simply guessing the mean quality.
- Even the best model has R2 of only 0.29, so about 71% of the variation in quality is not captured by these six chemical measurements.

## 7. Overfitting / underfitting check (train vs test)

In [ ]:
rows = []
for name, model in reg_models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    })

reg_overfit = pd.DataFrame(rows)
display(reg_overfit.round(4))

- **Decision Tree - severe overfitting:** train R2 = 0.999 but test R2 = -0.36. The unrestricted tree memorises the training wines.
- **Random Forest - overfitting (moderate):** train R2 = 0.90 against test R2 = 0.29. Averaging many trees makes it generalise far better than a single tree, but the gap is still large.
- **Linear Regression - mild underfitting:** train R2 = 0.293 and test R2 = 0.262 are close, so it is stable, but it is too simple to capture more than a limited share of the variation.

Possible improvements: limit tree depth (`max_depth`, `min_samples_leaf`), tune the forest, use cross-validation, add the remaining five chemical variables.

## 8. Which variables matter? Permutation importance (Random Forest)
Permutation importance measures how much the **test RMSE gets worse** when one variable is shuffled; a bigger value means the model relies on that variable more.

In [ ]:
perm = permutation_importance(
    rf_reg, X_test, y_test,
    n_repeats=10, random_state=42,
    scoring="neg_root_mean_squared_error",
)

reg_importance = pd.DataFrame({
    "Feature": X_reg.columns,
    "Importance": perm.importances_mean,
}).sort_values(by="Importance", ascending=False)

display(reg_importance.round(4))

plt.figure(figsize=(8, 5))
sns.barplot(data=reg_importance, x="Importance", y="Feature", hue="Feature", legend=False)
plt.title("Regression - Permutation Feature Importance (Random Forest)")
plt.xlabel("Increase in test RMSE when the variable is shuffled")
plt.ylabel("Variable")
plt.savefig("../outputs/charts/regression_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. OLS coefficients vs Random Forest importance
To compare on the same scale, a linear regression is fitted on **standardised** variables (z-scores). This gives the same *standardised OLS coefficients* as the statsmodels model in notebook 04.

In [ ]:
Z_X = (X_reg - X_reg.mean()) / X_reg.std()
Z_y = (y_reg - y_reg.mean()) / y_reg.std()
std_ols = LinearRegression().fit(Z_X, Z_y)

importance_comparison = pd.DataFrame({
    "Feature": features,
    "Standardized OLS Coefficient": std_ols.coef_,
    "RF Permutation Importance": reg_importance.set_index("Feature").loc[features, "Importance"].values,
})
display(importance_comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
sns.barplot(data=importance_comparison, x="Standardized OLS Coefficient", y="Feature",
            hue="Feature", legend=False, ax=axes[0])
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("OLS: standardised coefficient")
sns.barplot(data=importance_comparison, x="RF Permutation Importance", y="Feature",
            hue="Feature", legend=False, ax=axes[1])
axes[1].set_title("Random Forest: permutation importance")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("../outputs/charts/ols_vs_rf_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### Statistical association vs predictive importance

| Variable | Standardised OLS coefficient | RF permutation importance |
|---|---|---|
| alcohol | +0.339 | **0.185** |
| density | -0.339 | 0.030 |
| residual sugar | +0.332 | 0.062 |
| volatile acidity | -0.206 | 0.064 |
| pH | +0.124 | 0.032 |
| citric acid | +0.041 | 0.027 |

- **Alcohol** is the clear leader in the Random Forest (about three times the importance of the next variable) and has one of the largest OLS coefficients. It is the most dependable finding in the project.
- **Density and residual sugar** have OLS coefficients as large as alcohol (-0.34 and +0.33) but low predictive importance. Their coefficients have opposite signs and largely cancel each other out: because the two variables are highly collinear (VIF 11.2 and 5.5), OLS splits the shared information between them in an unstable way. The forest, which is judged on held-out accuracy, finds that they add little beyond alcohol.
- **Volatile acidity** is the second most important variable for prediction, and its OLS coefficient is moderate and stable (-0.21).

**The difference:** an OLS coefficient describes the *association between a variable and quality, holding the others constant, in the sample* (a statistical, explanatory view that can be distorted by collinearity). Permutation importance measures *how much the model's accuracy on unseen data depends on the variable* (a predictive view). A variable can be significant in OLS but add little to prediction, and the permutation importance of correlated variables is also shared between them. Both views are needed.

## 10. Save the regression comparison

In [ ]:
out = reg_comparison.copy()
out.insert(0, "Task", "Regression")
out.to_csv("../outputs/model_comparison.csv", index=False)
reg_comparison.to_csv("../outputs/regression_model_comparison.csv", index=False)
print("Saved: ../outputs/model_comparison.csv (regression rows)")

## Conclusion (regression)
The Random Forest is the best regressor (RMSE 0.762, R2 0.291), followed closely by Linear Regression (RMSE 0.778, R2 0.262), while the unrestricted Decision Tree overfits badly. For a purely predictive task, choose the Random Forest; if interpretability matters, Linear Regression gives almost the same accuracy with far simpler explanations. All models leave most of the variation unexplained, so predictions should be used as a guide rather than as a precise quality score.